# 07 — LLMOps: Deney Takibi, A/B Test, Model Versiyonlama

**İlan maddesi:** *"LLMOps altyapısı kurmak; deney takibi, model versiyonlama, A/B test
süreçleri ve izleme panelleri oluşturmak."*

Bu notebook üç bileşeni gösterir:
1. MLflow ile deney/eğitim takibi (04. ve 05. notebook'lardaki eğitimler zaten
   `report_to=["mlflow"]` ile otomatik loglanıyor)
2. İki model varyantını karşılaştıran istatistiksel A/B test
3. Streamlit izleme panelinin nasıl başlatılacağı

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## 1. MLflow deneylerini görüntüleme

Colab'da arka planda başlatıp `localtunnel` ile dışarı açıyoruz.

In [ ]:
get_ipython().system_raw("mlflow ui --backend-store-uri file:./mlruns --port 5000 &")
!npx --yes localtunnel --port 5000


## 2. A/B Test

Örnek: temel model vs. QLoRA+DPO modeli, aynı soru seti üzerinde groundedness skoruna göre.

In [ ]:
from src.llmops.ab_test import run_ab_test
from src.rag.rag_pipeline import answer
from src.xai.hallucination_check import groundedness_score
from src.config import MODEL_CONFIG

eval_questions = [
    "Bayraktar TB2 hangi görevlerde kullanılır?",
    "ASELSAN hangi alanlarda faaliyet gösterir?",
    "Elektronik harp nedir?",
]

def eval_base(q):
    return answer(q, model_path=MODEL_CONFIG.base_llm)

def eval_finetuned(q):
    return answer(q, model_path="models/dpo-adapter")  # 05. notebook çıktısı

def score_fn(question, result):
    contexts = [c["text"] for c in result["contexts"]]
    return groundedness_score(result["answer"], contexts) if contexts else 0.0

result = run_ab_test("base", "finetuned", eval_base, eval_finetuned, eval_questions, score_fn)
print(result)


## 3. İzleme paneli

FastAPI servisini (bkz. `src/serving/api.py`) çalıştırıp birkaç istek attıktan sonra:

In [ ]:
# Ayrı bir terminalde / hücrede:
# !uvicorn src.serving.api:app --host 0.0.0.0 --port 8000 &
# !streamlit run src/llmops/dashboard.py &
print("Panel için terminalde yukarıdaki komutları çalıştırın.")
